In [21]:
!pip install scikit-learn gensim nltk  PyPDF2

In [32]:
from google.colab import files
uploaded = files.upload()

Saving Python V3.pdf to Python V3.pdf


In [22]:
import pandas as pd
import PyPDF2

def extract_text_from_pdf(pdf_path):
    text = []
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text.append(page.extract_text())
    return text



Extracted 17 pages from the PDF.


In [22]:
#pdf = uploaded

pdf = '/content/Legal Aspects of Mergers and Acquisitions in India.pdf'
documents = extract_text_from_pdf(pdf)

print(f"Extracted {len(documents)} pages from the PDF.")

In [23]:
import nltk
import re
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [w for w in text.split() if w not in stop_words]
    return " ".join(tokens)

documents_clean = [preprocess(doc) for doc in documents]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(documents_clean)

lsa_model = TruncatedSVD(n_components=10, random_state=42)
lsa_topics = lsa_model.fit_transform(X)


In [25]:
terms = vectorizer.get_feature_names_out()

for i, comp in enumerate(lsa_model.components_):
    terms_comp = zip(terms, comp)
    sorted_terms = sorted(terms_comp, key=lambda x: x[1], reverse=True)[:10]
    print(f"Topic {i}: ", [t[0] for t in sorted_terms])


Topic 0:  ['sebi', 'section', 'act', 'regulations', 'india', 'mergers', 'company', 'rbi', 'trading', 'companies']
Topic 1:  ['sebi', 'trading', 'insider', 'information', 'sensitive', 'regulations', 'price', 'prohibition', 'unpublished', 'shares']
Topic 2:  ['banking', 'rbi', 'bank', 'reserve', 'section', 'regulation', 'india', 'banks', 'act', 'goods']
Topic 3:  ['tata', 'steel', 'air', 'zomato', 'hdfc', 'acquisition', 'rbi', 'blinkit', 'bank', 'bhushan']
Topic 4:  ['hdfc', 'tata', 'air', 'zomato', 'steel', 'idea', 'vodafone', 'blinkit', 'ltd', 'indias']
Topic 5:  ['stamp', 'steel', 'tata', 'fema', 'border', 'cross', 'bhushan', 'must', 'assets', 'duty']
Topic 6:  ['law', 'tata', 'boldest', 'burdens', 'chapters', 'continues', 'dynamic', 'gives', 'hinder', 'liabilities']
Topic 7:  ['stamp', 'fema', 'assets', 'duty', 'crore', 'idea', 'vodafone', 'adverse', 'appreciable', 'effect']
Topic 8:  ['integration', 'control', 'facebook', 'organization', 'supply', 'backward', 'chain', 'forward', 'ho

In [26]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

count_vectorizer = CountVectorizer(max_features=5000)
X_counts = count_vectorizer.fit_transform(documents_clean)


In [27]:
lda_model = LatentDirichletAllocation(
    n_components=10,
    random_state=42
)
lda_model.fit(X_counts)


LatentDirichletAllocation(random_state=42)

In [28]:
words = count_vectorizer.get_feature_names_out()

for idx, topic in enumerate(lda_model.components_):
    print(f"Topic {idx}: ",
          [words[i] for i in topic.argsort()[-10:]])


Topic 0:  ['law', 'bank', 'merger', 'acquisition', 'blinkit', 'hdfc', 'zomato', 'tata', 'air', 'india']
Topic 1:  ['must', 'business', 'periods', 'public', 'insiders', 'trading', 'information', 'sensitive', 'sebi', 'price']
Topic 2:  ['deals', 'one', 'grow', 'tax', 'market', 'legal', 'law', 'access', 'competition', 'company']
Topic 3:  ['notified', 'set', 'indian', 'business', 'acquisition', 'legal', 'mergers', 'corporate', 'sector', 'foreign']
Topic 4:  ['company', 'india', 'indian', 'companies', 'mergers', 'market', 'stamp', 'competition', 'act', 'merger']
Topic 5:  ['regulation', 'companies', 'bank', 'tax', 'banking', 'rbi', 'india', 'mergers', 'section', 'act']
Topic 6:  ['distribution', 'backward', 'diversification', 'forward', 'companies', 'businesses', 'supply', 'control', 'integration', 'company']
Topic 7:  ['different', 'without', 'hdfc', 'coca', 'acquisitions', 'public', 'mergers', 'acquisition', 'merger', 'company']
Topic 8:  ['cross', 'tata', 'regulations', 'companies', 'tr

In [29]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=10, random_state=42)
nmf.fit(X)

terms = vectorizer.get_feature_names_out()

/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


In [30]:
print("\nNMF Topics:\n")
for i, comp in enumerate(nmf.components_):
    words = [terms[j] for j in comp.argsort()[-10:]]
    print(f"Topic {i}: {words}")


NMF Topics:

Topic 0: ['approval', 'act', 'banks', 'section', 'india', 'regulation', 'reserve', 'bank', 'rbi', 'banking']
Topic 1: ['unpublished', 'public', 'prohibition', 'price', 'regulations', 'sensitive', 'information', 'insider', 'trading', 'sebi']
Topic 2: ['concentric', 'suppliers', 'businesses', 'company', 'diversification', 'forward', 'backward', 'integration', 'supply', 'control']
Topic 3: ['legal', 'amalgamation', 'certain', 'sections', 'mergers', 'income', 'competition', 'tax', 'act', 'section']
Topic 4: ['vodafone', 'introduction', 'examples', 'acquisitions', 'walmart', 'flipkart', 'takeover', 'businesses', 'economies', 'hdfc']
Topic 5: ['fema', 'blinkit', 'india', 'cross', 'border', 'bhushan', 'zomato', 'air', 'steel', 'tata']
Topic 6: ['chapters', 'rewards', 'rewrite', 'respect', 'liabilities', 'understand', 'burdens', 'realbe', 'use', 'law']
Topic 7: ['subscriber', 'telecommunications', 'jio', 'support', 'department', 'merger', 'base', 'telecom', 'vodafone', 'idea']
To